<a href="https://colab.research.google.com/github/SiyaBhasin/Celebal-Technologies-Assignments/blob/main/Week7_Siya_Bhasin/RAG_Document_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Document Question Answering System (RAG)

**Retrieval-Augmented Generation over a custom PDF document**

This notebook builds a complete RAG pipeline that answers questions grounded in a custom document (PDF), instead of relying only on a language model's internal knowledge.

**Pipeline:** Document Ingestion → Text Chunking → Embedding Creation → Vector Store (FAISS) → Query Processing → Context Retrieval → Answer Generation

**Stack used:**
- `langchain` + `langchain-community` — orchestration
- `sentence-transformers` (`all-MiniLM-L6-v2`) — embeddings (free, local, no API key)
- `FAISS` — vector store / similarity search
- **Google Gemini** (`gemini-2.5-flash-lite`) — language model for answer generation (free tier, requires a free API key)


## 1. Setup — Install Dependencies

In [2]:
!pip install -q langchain langchain-classic langchain-community langchain-text-splitters langchain-huggingface langchain-google-genai faiss-cpu pypdf sentence-transformers


## 2. Imports

In [3]:
import os
import time
import textwrap

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

try:
    from langchain_classic.chains import RetrievalQA
except ImportError:
    from langchain.chains import RetrievalQA

print("All imports successful.")


/tmp/ipykernel_4082/945660521.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


All imports successful.


## 3. Document Ingestion

Load a custom PDF document and convert it into raw text. Replace `PDF_PATH` below with the path to your own PDF (notes, resume, research paper, book chapter, etc.)


In [4]:
import os

PDF_PATH = "/content/(R17A1204) Artificial Intelligence.pdf"

if not os.path.exists(PDF_PATH):
    try:
        from google.colab import files
        print("PDF not found at the default path — please upload it now:")
        uploaded = files.upload()
        PDF_PATH = list(uploaded.keys())[0]
    except ImportError:
        raise FileNotFoundError(
            f"File not found: {PDF_PATH}. Place your PDF there, or update PDF_PATH."
        )

print(f"Using PDF: {PDF_PATH}")

loader = PyPDFLoader(PDF_PATH)
raw_docs = loader.load()

print(f"Loaded {len(raw_docs)} page(s) from '{PDF_PATH}'")
print("\n--- Preview of page 1 ---\n")
print(raw_docs[0].page_content[:500])


PDF not found at the default path — please upload it now:


Saving (R17A1204) Artificial Intelligence.pdf to (R17A1204) Artificial Intelligence.pdf
Using PDF: (R17A1204) Artificial Intelligence.pdf
Loaded 143 page(s) from '(R17A1204) Artificial Intelligence.pdf'

--- Preview of page 1 ---

Artificial Intelligence Page 1 
 
DIGITAL NOTES  
ON 
ARTIFICIAL INTELLIGENCE 
B.TECH IV YR / I SEM 
(2020-21) 
 
 
 
 
 
 
 
 
DEPARTMENT OF INFORMATION TECHNOLOGY 
MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY 
 (Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified) 
Maisammaguda, Dhulapally (Post Via. Hakimpet), Secunderabad – 500100


## 4. Text Chunking

The text is split into smaller overlapping chunks to improve retrieval accuracy. Smaller chunks retrieve more precisely; overlap prevents losing context at chunk boundaries.


In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)

chunks = text_splitter.split_documents(raw_docs)

print(f"Document split into {len(chunks)} chunks")
print("\n--- Example chunk ---\n")
print(chunks[0].page_content)


Document split into 545 chunks

--- Example chunk ---

Artificial Intelligence Page 1 
 
DIGITAL NOTES  
ON 
ARTIFICIAL INTELLIGENCE 
B.TECH IV YR / I SEM 
(2020-21) 
 
 
 
 
 
 
 
 
DEPARTMENT OF INFORMATION TECHNOLOGY 
MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY 
 (Autonomous Institution – UGC, Govt. of India) 
Recognized under 2(f) and 12 (B) of UGC ACT 1956 
(Affiliated to JNTUH, Hyderabad, Approved by AICTE - Accredited by NBA & NAAC – ‘A’ Grade - ISO 9001:2015 Certified)


## 5. Embedding Creation

Each chunk is converted into a vector representation (embedding) that captures its semantic meaning, using a free local sentence-transformer model (no API key needed).


In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

sample_vector = embedding_model.embed_query("This is a test sentence.")
print(f"Embedding model loaded. Vector dimension: {len(sample_vector)}")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embedding model loaded. Vector dimension: 384


## 6. Vector Database (FAISS)

Embeddings are stored in a FAISS vector store for fast similarity search. FAISS runs fully in-memory/on-disk locally — no server, no external service, no file-lock issues.


In [7]:
t0 = time.time()
vectorstore = FAISS.from_documents(chunks, embedding_model)
build_time = round(time.time() - t0, 2)

print(f"FAISS vector store built with {len(chunks)} chunks in {build_time}s")

vectorstore.save_local("faiss_index")
print("Vector store saved to ./faiss_index")


FAISS vector store built with 545 chunks in 39.69s
Vector store saved to ./faiss_index


## 7. Query Processing & Context Retrieval

Set up the retriever: it converts a user's question into an embedding and retrieves the most relevant chunks from the vector store.


In [8]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})

test_query = "What is this document about?"
retrieved = retriever.invoke(test_query)

print(f"Retrieved {len(retrieved)} chunks for query: '{test_query}'\n")
for i, doc in enumerate(retrieved, 1):
    print(f"--- Chunk {i} ---")
    print(textwrap.shorten(doc.page_content, width=200))
    print()


Retrieved 4 chunks for query: 'What is this document about?'

--- Chunk 1 ---
Artificial Intelligence Page 48

--- Chunk 2 ---
Artificial Intelligence Page 56 Algorithm:

--- Chunk 3 ---
Artificial Intelligence Page 67 Proof of Completeness FC derives every atomic sentence that is entailed by KB

--- Chunk 4 ---
to use the enormous rules that resulted. o Practical ignorance. Uncertain about a particular individual in the domain because all of the information necessary for that individual has not been [...]



## 8. Answer Generation — Google Gemini

Used Google Gemini (`gemini-2.5-flash-lite`) to generate the final answer.
Gemini's free tier needs an API key -
[aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)


In [9]:
import getpass
from langchain_google_genai import ChatGoogleGenerativeAI

GOOGLE_API_KEY = getpass.getpass("Enter your Google Gemini API key: ")

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.2,
    max_tokens=512,
)

print("Gemini LLM (gemini-2.5-flash-lite) ready.")


Enter your Google Gemini API key: ··········
Gemini LLM (gemini-2.5-flash-lite) ready.


## 9. Building the RAG Chain

Combined the retriever and the LLM into a single RetrievalQA chain: question → retrieve context → generate grounded answer.

Used a custom prompt that instructs the model to answer only from the
retrieved context, and to say so clearly if the answer isn't present —
this keeps answers grounded and avoids hallucination.


In [10]:
from langchain_core.prompts import PromptTemplate

qa_prompt = PromptTemplate(
    template=(
        "You are a precise document assistant. Answer the question using "
        "the context below. The context consists of several excerpts from "
        "the document — synthesize across all of them to form a complete "
        "answer, even if no single excerpt fully answers the question on "
        "its own. Write 2-4 complete, well-structured sentences.\n"
        "Only say \"I couldn't find the answer to that in the document.\" "
        "if the context is genuinely unrelated to the question.\n\n"
        "Context:\n{context}\n\n"
        "Question: {question}\n\n"
        "Answer:"
    ),
    input_variables=["context", "question"],
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    chain_type="stuff",
    chain_type_kwargs={"prompt": qa_prompt},
    return_source_documents=True,
)

print("RAG chain ready.")

RAG chain ready.


## 10. Ask Questions

Run questions through the full pipeline. Each answer is grounded in the retrieved chunks from your document, not the model's general knowledge.


In [11]:
def ask(question: str):
    t0 = time.time()
    result = qa_chain.invoke({"query": question})
    latency = round((time.time() - t0) * 1000)

    print(f"Q: {question}")
    print(f"A: {result['result'].strip()}")
    print(f"\n(answered in {latency} ms, using {len(result['source_documents'])} retrieved chunks)")
    print("\n--- Source chunks used ---")
    for i, doc in enumerate(result["source_documents"], 1):
        page = doc.metadata.get("page", "?")
        print(f"[{i}] (page {page}) {textwrap.shorten(doc.page_content, width=150)}")
    return result

_ = ask("What is the main idea of the document?")


Q: What is the main idea of the document?
A: The document discusses knowledge-based agents in Artificial Intelligence, which utilize a knowledge base and an inference mechanism to store, infer, and deduce actions. It also touches upon the foundational work in theoretical computer science by Goedel and Turing in the 1930s, which laid groundwork for AI. The concept of "practical ignorance" is also mentioned, referring to uncertainty due to incomplete information about an individual.

(answered in 1461 ms, using 4 retrieved chunks)

--- Source chunks used ---
[1] (page 119) to use the enormous rules that resulted. o Practical ignorance. Uncertain about a particular individual in the domain because all of the [...]
[2] (page 4) problems on their own. History of AI: Important research that laid the groundwork for AI:  In 1931, Goedel layed the foundation of Theoretical [...]
[3] (page 47) Artificial Intelligence Page 48
[4] (page 62) Artificial Intelligence Page 63 UNIT III Knowledge Based

In [12]:
_ = ask("Summarize the key points in this document.")


Q: Summarize the key points in this document.
A: The provided document excerpts indicate that "Artificial Intelligence" is a subject offered in the IV Year B.Tech IT program at Malla Reddy College of Engineering & Technology, specifically as course (R17A1204). The course objectives are designed to equip students with certain capabilities. Additionally, the document touches upon the concept of an "Algorithm" and a "Proof of Completeness" in the context of Artificial Intelligence, where FC derives every atomic sentence entailed by the knowledge base.

(answered in 953 ms, using 4 retrieved chunks)

--- Source chunks used ---
[1] (page 47) Artificial Intelligence Page 48
[2] (page 55) Artificial Intelligence Page 56 Algorithm:
[3] (page 1) Artificial Intelligence Page 2 MALLA REDDY COLLEGE OF ENGINEERING & TECHNOLOGY Department of Information Technology IV YearB.TechIT–ISem L T [...]
[4] (page 66) Artificial Intelligence Page 67 Proof of Completeness FC derives every atomic sentence that 

In [13]:
_ = ask("Tell me about history of ai in short from the pdf")


Q: Tell me about history of ai in short from the pdf
A: The history of Artificial Intelligence is rooted in foundational work in theoretical computer science, with Goedel laying groundwork in 1931 and Turing reformulating these results in 1936. The term "Artificial Intelligence" was coined by John McCarthy in 1956, and by the 1990s, significant advancements were seen across various AI domains, including machine learning and natural language understanding. A notable milestone occurred in 1997 when Deep Blue defeated the World Chess Champion.

(answered in 786 ms, using 4 retrieved chunks)

--- Source chunks used ---
[1] (page 4) problems on their own. History of AI: Important research that laid the groundwork for AI:  In 1931, Goedel layed the foundation of Theoretical [...]
[2] (page 4) Artificial Intelligence Page 5 UNIT I: Introduction:  Artificial Intelligence is concerned with the design of intelligence in an artificial [...]
[3] (page 4)  AI program will demonstrate a high leve

In [14]:
_ = ask("What is supervised learning? answer from the pdf")

Q: What is supervised learning? answer from the pdf
A: Supervised learning involves an agent learning a function that maps inputs to outputs by observing example input-output pairs. This process uses a training set of known input-output pairs to discover a hypothesis function that approximates the true underlying function. The accuracy of this learned function is then evaluated using a separate test set of examples.

(answered in 1659 ms, using 4 retrieved chunks)

--- Source chunks used ---
[1] (page 136) a large collection of unlabelled examples SUPERVISED LEARNING Given a training set of N example input–output pairs (x1, y1), (x2, y2), . . . [...]
[2] (page 136) that are distinct from the training set. Conditional Probability Distribution : the function f is stochastic—it is not strictly a function of x, [...]
[3] (page 136) learning o In unsupervised learning the agent learns patterns in the input even though no explicit feedback is supplied o reinforcement learning [...]
[4] (page

## 11. Validation Log

Run a small suite of sample questions to validate retrieval quality and measure latency across the pipeline.


In [15]:
validation_questions = [
    "What is the main topic of this document?",
    "What conclusions does the document reach?",
    "Are there any specific numbers, dates, or names mentioned?",
    "What recommendations or next steps are described?",
]

validation_log = []
for q in validation_questions:
    t0 = time.time()
    result = qa_chain.invoke({"query": q})
    latency = round((time.time() - t0) * 1000)
    validation_log.append({
        "question": q,
        "answer": result["result"].strip(),
        "chunks_used": len(result["source_documents"]),
        "latency_ms": latency,
    })

import pandas as pd
pd.DataFrame(validation_log)


,question,answer,chunks_used,latency_ms
0,What is the main topic of this document?,The main topic of this document is Artificial ...,4,745
1,What conclusions does the document reach?,The document explains that a sentence is satis...,4,757
2,"Are there any specific numbers, dates, or name...","Yes, the document mentions the year 1974 in re...",4,857
3,What recommendations or next steps are described?,"The document suggests a ""plan first, schedule ...",4,1711


## 12. System Metrics Summary


In [16]:
print("=== RAG Pipeline Summary ===")
print(f"Source document       : {PDF_PATH}")
print(f"Pages loaded           : {len(raw_docs)}")
print(f"Chunks created          : {len(chunks)}")
print(f"Chunk size / overlap    : 500 / 50")
print(f"Embedding model         : sentence-transformers/all-MiniLM-L6-v2")
print(f"Embedding dimension     : {len(sample_vector)}")
print(f"Vector store             : FAISS ({len(chunks)} vectors)")
print(f"LLM                      : Google Gemini (gemini-2.5-flash-lite)")
print(f"Retriever top-k          : 4")
print(f"Validation questions run : {len(validation_log)}")
avg_latency = round(sum(v['latency_ms'] for v in validation_log) / len(validation_log))
print(f"Average answer latency   : {avg_latency} ms")


=== RAG Pipeline Summary ===
Source document       : (R17A1204) Artificial Intelligence.pdf
Pages loaded           : 143
Chunks created          : 545
Chunk size / overlap    : 500 / 50
Embedding model         : sentence-transformers/all-MiniLM-L6-v2
Embedding dimension     : 384
Vector store             : FAISS (545 vectors)
LLM                      : Google Gemini (gemini-2.5-flash-lite)
Retriever top-k          : 4
Validation questions run : 4
Average answer latency   : 1018 ms


## Conclusion

This notebook demonstrates a complete Retrieval-Augmented Generation pipeline:

1. **Document Ingestion** — loaded a custom PDF
2. **Text Chunking** — split into overlapping chunks for accurate retrieval
3. **Embedding Creation** — converted chunks into semantic vectors
4. **Vector Database** — stored embeddings in FAISS for fast similarity search
5. **Query Processing & Retrieval** — converted questions into embeddings and retrieved relevant chunks
6. **Answer Generation** — generated grounded answers using a local language model

### Key Learnings
- How RAG systems combine retrieval and generation to ground answers in real data
- The importance of chunk size and overlap for retrieval quality
- Working with embeddings and vector databases (FAISS)
- Connecting a retrieval pipeline to a hosted LLM (Google Gemini) via LangChain